# Workflow lifecycle smoke check

This fixture validates planning, review, execution, and provenance without full-data fitting.

In [1]:
from pathlib import Path
import sys
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'integrated_system').is_dir())
sys.path.insert(0, str(project_root / 'integrated_system' / 'src'))
from pfas_workflow.catalog import ArtifactCatalog
from pfas_workflow.executor import WorkflowExecutor
from pfas_workflow.models import ApprovedExecution, BiologicalRequest, HumanReview, ReviewDecision
from pfas_workflow.planner import RuleBasedPlanner
catalog = ArtifactCatalog(project_root)
plan = RuleBasedPlanner(catalog).plan(BiologicalRequest(question='Summarize PFOS differential expression.'))
assert plan.accepted and plan.citations
review = HumanReview(decision=ReviewDecision.APPROVED, reviewer='ci scientific review', plan_digest=plan.review_digest())
result = WorkflowExecutor(catalog).execute(ApprovedExecution(plan=plan, review=review))
assert result.records[0]['chemical'] == 'PFOS'
assert any('result_input' in item.roles for item in result.provenance.values())
result.model_dump(mode='json')

{'execution_id': '487777380c1471018e94',
 'plan_digest': '20f56952ddbac77e29ee1a3aaefd59097f361c197bc11e5d614bc2dda7387e73',
 'analysis': 'deg_summary',
 'records': [{'chemical': 'PFOS',
   'genes_tested': 13852,
   'fdr_threshold': 0.05,
   'genes_detected_at_threshold': 190}],
 'provenance': {'Data/DEGs/PFOSvsControl_DGE_results.csv': {'sha256': 'b538146b1da3ce8978a2121212988049e03fe4197da5ec66a09d61cd4cbf11f0',
   'roles': ['result_input', 'planning_evidence']},
  'K-spaces Analysis/FINDINGS.md': {'sha256': 'f4eb6ceab5842b652638a9eddcea16f40551216d5cd14f164188c916283f5db5',
   'roles': ['planning_evidence']}},
 'findings': [{'code': 'contrast_scope',
   'severity': 'warning',
   'message': 'FDR is controlled within each contrast, not across all ten contrasts.'},
  {'code': 'expert_interpretation',
   'severity': 'info',
   'message': 'The approved computation completed; biological interpretation remains subject to domain-expert review.'}],
 'review': {'decision': 'approved',
  'revi